In [1]:
from pathlib import Path
import torch
import torch.nn as nn
from loguru import logger
import warnings
warnings.simplefilter("ignore", UserWarning)

Let's use the mads_datasets package (see [github](https://github.com/raoulg/mads_datasets) for more details) which I created for these lessons to give everyone easy access to the datasets we use for training.

In [2]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import BasePreprocessor

for dataset in DatasetType:
    print(dataset)

DatasetType.FLOWERS
DatasetType.IMDB
DatasetType.GESTURES
DatasetType.FASHION
DatasetType.SUNSPOTS
DatasetType.IRIS
DatasetType.PENGUINS
DatasetType.FAVORITA
DatasetType.SECURE


There are a few datasets. For images, we can use FLOWERS (~3000 photos of flowers in 5 categories) and FASHION (60k fashion icons 28x28 pixels big).

Lets start with our good'ol MNIST.

In [3]:

fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
batchsize = 64
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

2026-03-01 05:32:17.389 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /Users/stevenbontius/.cache/mads_datasets/fashionmnist
2026-03-01 05:32:17.389 | INFO     | mads_datasets.base:download_data:124 - File already exists at /Users/stevenbontius/.cache/mads_datasets/fashionmnist/fashionmnist.pt


We can obtain an item:

In [4]:
x, y = next(iter(trainstreamer))
x.shape, y.shape

(torch.Size([64, 1, 28, 28]), torch.Size([64]))

The image follows the channels-first convention: (channel, width, height). The label is an integer.

Let's re-use the model we had:

In [5]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

Using MPS


In [6]:
from torch import nn
print(f"Using {device} device")

# Define model
class CNN(nn.Module):
    def __init__(self, filters, units1, units2, input_size=(32, 1, 28, 28)):
        super().__init__()
        self.in_channels = input_size[1]
        self.input_size = input_size
        self.filters = filters
        self.units1 = units1
        self.units2 = units2

        self.convolutions = nn.Sequential(
            nn.Conv2d(self.in_channels, filters, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        activation_map_size = self._conv_test(input_size)
        logger.info(f"Aggregating activationmap with size {activation_map_size}")
        self.agg = nn.AvgPool2d(activation_map_size)

        self.dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, 10)
        )

    def _conv_test(self, input_size = (32, 1, 28, 28)):
        x = torch.ones(input_size)
        x = self.convolutions(x)
        return x.shape[-2:]

    def forward(self, x):
        x = self.convolutions(x)
        x = self.agg(x)
        logits = self.dense(x)
        return logits

model = CNN(filters=32, units1=128, units2=64).to("cpu")

2026-03-01 05:32:17.470 | INFO     | __main__:__init__:27 - Aggregating activationmap with size torch.Size([2, 2])


Using mps device


In [7]:
from mltrainer.imagemodels import CNNConfig, CNNblocks

In [8]:
config = CNNConfig(
    matrixshape = (28, 28), # every image is 28x28
    batchsize = batchsize,
    input_channels = 1, # we have black and white images, so only one channel
    hidden = 32, # number of filters
    kernel_size = 3, # kernel size of the convolution
    maxpool = 3, # kernel size of the maxpool
    num_layers = 4, # we will stack 4 Convolutional blocks, each with two Conv2d layers
    num_classes = 10,
)

In [9]:
model = CNNblocks(config)
model.config

Calculated matrix size: 9
Caluclated flatten size: 288


{'matrixshape': (28, 28),
 'batchsize': 64,
 'input_channels': 1,
 'hidden': 32,
 'kernel_size': 3,
 'maxpool': 3,
 'num_layers': 4,
 'num_classes': 10}

In [10]:
from torchinfo import summary
summary(model, input_size=(32, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
CNNblocks                                [32, 10]                  --
├─ModuleList: 1-1                        --                        --
│    └─ConvBlock: 2-1                    [32, 32, 28, 28]          --
│    │    └─Sequential: 3-1              [32, 32, 28, 28]          9,568
│    └─ConvBlock: 2-2                    [32, 32, 28, 28]          --
│    │    └─Sequential: 3-2              [32, 32, 28, 28]          18,496
│    └─ReLU: 2-3                         [32, 32, 28, 28]          --
│    └─MaxPool2d: 2-4                    [32, 32, 9, 9]            --
│    └─ConvBlock: 2-5                    [32, 32, 9, 9]            --
│    │    └─Sequential: 3-3              [32, 32, 9, 9]            18,496
│    └─ReLU: 2-6                         [32, 32, 9, 9]            --
│    └─ConvBlock: 2-7                    [32, 32, 9, 9]            --
│    │    └─Sequential: 3-4              [32, 32, 9, 9]            18,496


And set up the optimizer, loss and accuracy.

In [11]:
import torch.optim as optim
from mltrainer import metrics
optimizer = optim.Adam
loss_fn = torch.nn.CrossEntropyLoss()
accuracy = metrics.Accuracy()

In [12]:
yhat = model(x.to("cpu"))
accuracy(y.to("cpu"), yhat)

0.109375

In [13]:
from mltrainer import metrics, Trainer, TrainerSettings, ReportTypes
settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="demo",
    train_steps=100,
    valid_steps=100,
    reporttypes=[ReportTypes.TOML],
)

In [14]:
trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optimizer,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
trainer.loop()

2026-03-01 05:32:17.733 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to demo/20260301-053217
2026-03-01 05:32:18.628 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 100/100 [00:01<00:00, 69.67it/s]
2026-03-01 05:32:20.835 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.0802 test 1.5376 metric ['0.4425']
100%|██████████| 100/100 [00:00<00:00, 125.26it/s]
2026-03-01 05:32:21.904 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 1.0687 test 0.8394 metric ['0.6803']
100%|██████████| 100/100 [00:01<00:00, 89.15it/s]
2026-03-01 05:32:23.454 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.7458 test 0.7280 metric ['0.7152']
100%|██████████| 3/3 [00:04<00:00,  1.60s/it]


# MLflow
MLflow is an open-source platform designed to manage the entire Machine Learning (ML) lifecycle, including experimentation, reproducibility, deployment, and governance. It provides a set of APIs and tools to streamline ML workflows, making it easier to track experiments, package code, manage model versions, and deploy models.

Reasons to use MLflow over TensorBoard, gin-config, or Ray:

- End-to-end ML lifecycle management: While TensorBoard focuses on visualizing model training metrics and gin-config on hyperparameter configuration, MLflow covers a broader range of tasks, such as experiment tracking, model packaging, and deployment.

- Framework agnostic: MLflow is not tied to a specific ML framework, making it suitable for projects using different libraries or even multiple libraries.

- Model Registry: MLflow provides a centralized model registry, allowing you to version, track, and manage your models, which is not available in TensorBoard or gin-config.

- Deployment support: MLflow facilitates model deployment to various platforms, such as local, cloud, or Kubernetes environments, whereas TensorBoard and gin-config are not built for deployment tasks.

- Integration with other tools: MLflow integrates with popular tools and platforms like Databricks, AWS, and Azure, making it easy to incorporate into existing workflows.

However, the choice between MLflow and other tools like TensorBoard, gin-config, or Ray depends on your specific use case and the scope of the ML workflow you want to manage.

In [15]:
experiment_path = "mlflow_test2"

In [16]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(experiment_path)

2026/03/01 05:32:23 INFO mlflow.tracking.fluent: Experiment with name 'mlflow_test2' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/stevenbontius/Documents/Projects/MADS/MADS-ML-StevenB/mlruns/3', creation_time=1772339543896, experiment_id='3', last_update_time=1772339543896, lifecycle_stage='active', name='mlflow_test2', tags={}>

In the code above, we set the MLflow tracking URI to a local SQLite database file. This is done to configure the storage location for MLflow's experiment tracking data, such as metrics, parameters, and artifacts. By specifying a SQLite database, we enable a lightweight and easy-to-use storage solution for tracking the experiments and their associated information.

The line mlflow.set_experiment("mnist_convolutions") sets the active MLflow experiment to "mnist_convolutions". This is useful for organizing and grouping your runs, as it allows you to associate the upcoming ML training runs with a specific experiment name, making it easier to search, compare, and analyze the results later.

In [17]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

We import functions and classes from the hyperopt library to perform hyperparameter optimization. This library helps us find the best hyperparameter values for our machine learning model by searching through a defined search space and using optimization algorithms like Tree-structured Parzen Estimator (TPE). The goal is to improve our model's performance by tuning its hyperparameters.

Advantages of TPE:

- Model-based approach: TPE is a Bayesian optimization method that models the objective function as a probability distribution. It learns from previous evaluations to decide which points in the search space to explore next, making it more efficient in finding optimal hyperparameters.

- Exploration-exploitation trade-off: TPE balances the trade-off between exploration (searching in new regions of the search space) and exploitation (refining around the current best points). This can lead to better results in problems with complex search spaces.

- Continuous hyperparameter optimization: TPE can handle continuous hyperparameters more naturally, as it builds a probability model to estimate the performance for any given point in the search space.

Lets set up an objective function and start logging some usefull things we might want to track:

In [18]:
modeldir = Path("models").resolve()
if not modeldir.exists():
    modeldir.mkdir()
    print(f"Created {modeldir}")

In [ ]:
import torch.optim as optim
from mltrainer import metrics, Trainer, TrainerSettings, ReportTypes
from datetime import datetime

# Define the hyperparameter search space
settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=modeldir,
    train_steps=100,
    valid_steps=100,
    reporttypes=[ReportTypes.MLFLOW, ReportTypes.TOML],
)


# Define the objective function for hyperparameter optimization
def objective(params):
    # Start a new MLflow run for tracking the experiment
    with mlflow.start_run():
        # Set MLflow tags to record metadata about the model and developer
        mlflow.set_tag("model", "convnet")
        mlflow.set_tag("dev", "raoul")
        # Log hyperparameters to MLflow
        mlflow.log_params(params)
        mlflow.log_param("batchsize", f"{batchsize}")


        # Initialize the optimizer, loss function, and accuracy metric
        optimizer = optim.Adam
        loss_fn = torch.nn.CrossEntropyLoss()
        accuracy = metrics.Accuracy()
        config = CNNConfig(
            matrixshape = (28, 28), # every image is 28x28
            batchsize = batchsize,
            input_channels = 1, # we have black and white images, so only one channel
            hidden = params["filters"], # number of filters
            kernel_size = 3, # kernel size of the convolution
            maxpool = 3, # kernel size of the maxpool
            num_layers = 4, # we will stack 4 Convolutional blocks, each with two Conv2d layers
            num_classes = 10,
        )

        # Instantiate the CNN model with the given hyperparameters
        model = CNNblocks(config)
        # Train the model using a custom train loop
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optimizer,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        trainer.loop()

        # Save the trained model with a timestamp
        tag = datetime.now().strftime("%Y%m%d-%H%M")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

        # Log the saved model as an artifact in MLflow
        mlflow.log_artifact(local_path=modelpath, artifact_path="pytorch_models")
        return {'loss' : trainer.test_loss, 'status': STATUS_OK}

See https://hyperopt.github.io/hyperopt/getting-started/search_spaces/ for more information about searchspaces for hyperopt

In [20]:
search_space = {
    'filters' : scope.int(hp.quniform('filters', 16, 128, 8)),
    'kernel_size' : scope.int(hp.quniform('kernel_size', 2, 5, 1)),
    'num_layers' : scope.int(hp.quniform('num_layers', 1, 10, 1)),
}

We define a search space for hyperparameter optimization using Hyperopt. The search space specifies the range and distribution of hyperparameters to explore during the optimization process. This is crucial for finding the optimal set of hyperparameters that yield the best performance for the machine learning model. The search space defined here includes the number of filters in the convolutional layers, and the number of units in two fully connected layers, allowing Hyperopt to find the best combination within the given ranges.


Now, finally, let us perform the hyperparameter search using the fmin function from hyperopt. The function takes the following arguments:

- `fn=objective`: The objective function to minimize, which is defined earlier to train the model and return the test loss.
- `space=search_space`: The search space defined earlier, containing the range of hyperparameters to explore.
- `algo=tpe.suggest`: The optimization algorithm to use, in this case, the Tree-structured Parzen Estimator (TPE) method.
- `max_evals=10`: The maximum number of function evaluations, i.e., the maximum number of hyperparameter combinations to try.
- `trials=Trials()`: A Trials object to store the results of each evaluation.

The fmin function searches for the best hyperparameters within the given search space using the TPE algorithm, aiming to minimize the objective function (test loss). Once the optimization process is completed, the best hyperparameters found are stored in the best_result variable.

In [21]:
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=3,
    trials=Trials()
)

Calculated matrix size: 9                            
Caluclated flatten size: 216                         
  0%|          | 0/3 [00:00<?, ?trial/s, best loss=?]

2026-03-01 05:32:24.589 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to /Users/stevenbontius/Documents/Projects/MADS/MADS-ML-StevenB/models/20260301-053224
2026-03-01 05:32:24.608 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:01<00:00, 70.77it/s]
2026-03-01 05:32:26.295 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.7689 test 1.1004 metric ['0.5825']
100%|##########| 100/100 [00:00<00:00, 105.26it/s]
2026-03-01 05:32:27.767 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.9667 test 0.8576 metric ['0.6766']
100%|##########| 100/100 [00:01<00:00, 84.24it/s]
2026-03-01 05:32:29.278 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.7902 test 0.7705 metric ['0.7184']
100%|##########| 3/3 [00:04<00:00,  1.56s/it]


Calculated matrix size: 9                                                      
Caluclated flatten size: 432                                                   
 33%|███▎      | 1/3 [00:04<00:09,  4.89s/trial, best loss: 0.7704795223474502]

2026-03-01 05:32:29.307 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to /Users/stevenbontius/Documents/Projects/MADS/MADS-ML-StevenB/models/20260301-053229
2026-03-01 05:32:29.313 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:01<00:00, 52.70it/s]
2026-03-01 05:32:31.661 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.6232 test 0.9335 metric ['0.6602']
100%|##########| 100/100 [00:01<00:00, 64.61it/s]
2026-03-01 05:32:33.681 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.7910 test 0.7252 metric ['0.7048']
100%|##########| 100/100 [00:01<00:00, 59.39it/s]
2026-03-01 05:32:35.821 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.7035 test 0.6405 metric ['0.7617']
100%|##########| 3/3 [00:06<00:00,  2.17s/it]


Calculated matrix size: 9                                                      
Caluclated flatten size: 648                                                   
 67%|██████▋   | 2/3 [00:11<00:05,  5.86s/trial, best loss: 0.6405495747923851]

2026-03-01 05:32:35.849 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to /Users/stevenbontius/Documents/Projects/MADS/MADS-ML-StevenB/models/20260301-053235
2026-03-01 05:32:35.857 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:03<00:00, 29.03it/s]
2026-03-01 05:32:40.079 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.6442 test 1.0321 metric ['0.6180']
100%|##########| 100/100 [00:02<00:00, 34.12it/s]
2026-03-01 05:32:43.787 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.8012 test 0.7520 metric ['0.7089']
100%|##########| 100/100 [00:02<00:00, 34.12it/s]
2026-03-01 05:32:47.481 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.7033 test 0.6825 metric ['0.7453']
100%|##########| 3/3 [00:11<00:00,  3.87s/it]


100%|██████████| 3/3 [00:23<00:00,  7.70s/trial, best loss: 0.6405495747923851]


After running this, you can look at the best_result

In [22]:
best_result

{'filters': np.float64(48.0),
 'kernel_size': np.float64(5.0),
 'num_layers': np.float64(7.0)}

# MLflow GUI
MLflow has a really great dashboard.
you can see it with the command:
```bash
mlflow server \
    --backend-store-uri sqlite:///mlflow.db \
    --host 127.0.0.1 \ 
    --port 5000 \
```

The `--backend-store-uri` argument specifies the location of the SQLite database file, which is used to store experiment metadata, such as parameters, metrics, and artifacts. 
We have initialized the experiment with `mlflow.set_tracking_uri("sqlite:///mlflow.db")`, so `mlflow.db` is the location we need. 

`--host` tells the server we use our own machine (localhost), and `--port` specifies the port number on which the server will listen for incoming requests. In this case, we are using port 5000. Sometimes, you can have conflicts on a specific port and it could help to change the port (eg to 5001)

Note that on a windows machine, the `\` gives errors (because, why not, right) so for windows you might need to remove the `\` and put everything on a single line.

I have created a `Makefile` to automate these commands, but if it doesnt work (again, because you are on windows for example) you can just type the command by hand in your terminal.

After starting this up, go to `http://127.0.0.1:5000` in your browser. You should see the MLflow UI, where you can explore your experiments, runs, and metrics. The UI provides a user-friendly way to visualize and compare different runs, making it easier to analyze the results of your hyperparameter optimization and model training.